# Tugas 2 - Perbandingan CountVectorizer dan TF-IDF

**Nama:** Radit Firansah  
**NIM:** 244107020196

Membandingkan performa Multinomial Naive Bayes dengan fitur CountVectorizer dan TF-IDF pada data spam.csv.

## 1. Load dan Preprocessing Data

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

df = pd.read_csv('../data/spam.csv', encoding='latin-1')

# Drop 3 kolom kosong dan rename kolom
df = df.drop(df.iloc[:, 2:], axis=1)
df = df.rename(columns={'v1': 'Labels', 'v2': 'SMS'})

# Encode label
df['Labels'] = df['Labels'].map({'spam': 1, 'ham': 0})

X = df['SMS'].values
y = df['Labels'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=50)

## 2. Model dengan CountVectorizer (stop_words='english')

In [2]:
cv = CountVectorizer(stop_words='english')
X_train_cv = cv.fit_transform(X_train)
X_test_cv = cv.transform(X_test)

mnb_cv = MultinomialNB()
mnb_cv.fit(X_train_cv, y_train)

acc_train_cv = accuracy_score(y_train, mnb_cv.predict(X_train_cv))
acc_test_cv = accuracy_score(y_test, mnb_cv.predict(X_test_cv))

print('Jumlah fitur CountVectorizer:', len(cv.get_feature_names_out()))
print(f'Akurasi train CountVectorizer: {acc_train_cv}')
print(f'Akurasi test  CountVectorizer: {acc_test_cv}')

Jumlah fitur CountVectorizer: 7466
Akurasi train CountVectorizer: 0.9946152120260264
Akurasi test  CountVectorizer: 0.9829596412556054


## 3. Model dengan TF-IDF (stop_words='english')

In [3]:
tfidf = TfidfVectorizer(stop_words='english')
X_train_tf = tfidf.fit_transform(X_train)
X_test_tf = tfidf.transform(X_test)

mnb_tf = MultinomialNB()
mnb_tf.fit(X_train_tf, y_train)

acc_train_tf = accuracy_score(y_train, mnb_tf.predict(X_train_tf))
acc_test_tf = accuracy_score(y_test, mnb_tf.predict(X_test_tf))

print('Jumlah fitur TF-IDF:', len(tfidf.get_feature_names_out()))
print(f'Akurasi train TF-IDF: {acc_train_tf}')
print(f'Akurasi test  TF-IDF: {acc_test_tf}')

Jumlah fitur TF-IDF: 7466
Akurasi train TF-IDF: 0.9842943684092439
Akurasi test  TF-IDF: 0.9605381165919282


## 4. Perbandingan Hasil

In [4]:
hasil = pd.DataFrame({
    'Fitur': ['CountVectorizer', 'TF-IDF'],
    'Akurasi Train': [acc_train_cv, acc_train_tf],
    'Akurasi Test': [acc_test_cv, acc_test_tf],
})
hasil

,Fitur,Akurasi Train,Akurasi Test
0,CountVectorizer,0.994615,0.982960
1,TF-IDF,0.984294,0.960538


## 5. Kesimpulan

Berdasarkan tabel perbandingan di atas, **CountVectorizer** memberikan akurasi test yang lebih baik (0.9830) dibandingkan **TF-IDF** (0.9605) pada dataset `spam.csv` ini.

Pada kasus ini, **CountVectorizer dengan `stop_words='english'` lebih cocok** digunakan. Hal ini karena data SMS relatif pendek dan kata-kata pembeda spam (misalnya "free", "win", "claim", "txt") sangat khas dan muncul berulang. Frekuensi kemunculan kata mentah sudah cukup informatif bagi Multinomial Naive Bayes, sehingga pembobotan TF-IDF yang menurunkan bobot kata sering muncul justru sedikit mengurangi sinyal pembeda tersebut. Dengan demikian, untuk dataset `spam.csv`, fitur terbaik adalah **CountVectorizer**.